In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All required libraries imported")

All required libraries imported


In [2]:
df = pd.read_csv("../data/cleaned_transactions.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Data loaded successfully")
print("Shape:", df.shape)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

Data loaded successfully
Shape: (42750, 9)
Date range: 2023-01-01 08:00:00 to 2024-09-05 18:00:00


In [3]:
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

df["CohortMonth"] = (
    df.groupby("Customer ID")["InvoiceDate"]
      .transform("min")
      .dt.to_period("M")
      .dt.to_timestamp()
)

print("Cohort months created successfully")
print(df[["Customer ID", "InvoiceDate", "InvoiceMonth", "CohortMonth"]].head())

Cohort months created successfully
   Customer ID         InvoiceDate InvoiceMonth CohortMonth
0      11095.0 2023-02-20 10:00:00   2023-02-01  2023-01-01
1      10248.0 2023-08-14 15:00:00   2023-08-01  2023-05-01
2      10793.0 2023-08-23 17:00:00   2023-08-01  2023-07-01
3      10095.0 2023-10-02 08:00:00   2023-10-01  2023-05-01
4      10718.0 2023-04-11 15:00:00   2023-04-01  2023-02-01


In [4]:
df["CohortIndex"] = (
    (df["InvoiceMonth"].dt.year - df["CohortMonth"].dt.year) * 12
    + (df["InvoiceMonth"].dt.month - df["CohortMonth"].dt.month)
    + 1
)

print("Cohort index created successfully")
print(df[["Customer ID", "InvoiceMonth", "CohortMonth", "CohortIndex"]].head())
print("\nCohort Index range:", df["CohortIndex"].min(), "to", df["CohortIndex"].max())

Cohort index created successfully
   Customer ID InvoiceMonth CohortMonth  CohortIndex
0      11095.0   2023-02-01  2023-01-01            2
1      10248.0   2023-08-01  2023-05-01            4
2      10793.0   2023-08-01  2023-07-01            2
3      10095.0   2023-10-01  2023-05-01            6
4      10718.0   2023-04-01  2023-02-01            3

Cohort Index range: 1 to 8


In [5]:
cohort_data = (
    df.groupby(["CohortMonth", "CohortIndex"])["Customer ID"]
      .nunique()
      .reset_index()
)

cohort_counts = cohort_data.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="Customer ID"
)

print("Cohort customer counts calculated successfully")
print(cohort_counts)


Cohort customer counts calculated successfully
CohortIndex      1      2      3      4      5      6      7     8
CohortMonth                                                       
2023-01-01   703.0  696.0  644.0  553.0  431.0  303.0  154.0  48.0
2023-02-01    92.0   92.0   92.0   91.0   92.0   90.0   90.0  13.0
2023-03-01    74.0   74.0   74.0   74.0   74.0   74.0   67.0   2.0
2023-04-01    76.0   76.0   76.0   75.0   76.0   76.0   54.0   NaN
2023-05-01    62.0   62.0   62.0   62.0   62.0   62.0   27.0   NaN
2023-06-01    50.0   50.0   50.0   50.0   50.0   48.0   11.0   NaN
2023-07-01    34.0   34.0   34.0   34.0   34.0   32.0    5.0   NaN
2023-08-01    38.0   38.0   38.0   38.0   38.0   37.0    NaN   NaN
2023-09-01    27.0   27.0   27.0   27.0   27.0   19.0    NaN   NaN
2023-10-01    15.0   15.0   15.0   15.0   15.0    6.0    NaN   NaN
2023-11-01     9.0    9.0    9.0    9.0    9.0    2.0    NaN   NaN
2023-12-01     5.0    5.0    5.0    5.0    5.0    NaN    NaN   NaN
2024-01-01     

In [6]:
cohort_retention = cohort_counts.div(
    cohort_counts[1], axis=0
)

print("Cohort retention calculated successfully")
print((cohort_retention * 100).round(2))

Cohort retention calculated successfully
CohortIndex      1      2       3       4       5       6      7      8
CohortMonth                                                            
2023-01-01   100.0   99.0   91.61   78.66   61.31   43.10  21.91   6.83
2023-02-01   100.0  100.0  100.00   98.91  100.00   97.83  97.83  14.13
2023-03-01   100.0  100.0  100.00  100.00  100.00  100.00  90.54   2.70
2023-04-01   100.0  100.0  100.00   98.68  100.00  100.00  71.05    NaN
2023-05-01   100.0  100.0  100.00  100.00  100.00  100.00  43.55    NaN
2023-06-01   100.0  100.0  100.00  100.00  100.00   96.00  22.00    NaN
2023-07-01   100.0  100.0  100.00  100.00  100.00   94.12  14.71    NaN
2023-08-01   100.0  100.0  100.00  100.00  100.00   97.37    NaN    NaN
2023-09-01   100.0  100.0  100.00  100.00  100.00   70.37    NaN    NaN
2023-10-01   100.0  100.0  100.00  100.00  100.00   40.00    NaN    NaN
2023-11-01   100.0  100.0  100.00  100.00  100.00   22.22    NaN    NaN
2023-12-01   100.0  100

In [7]:
plt.figure(figsize=(12, 7))

sns.heatmap(
    cohort_retention * 100,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100
)

plt.title("Customer Cohort Retention Analysis")
plt.xlabel("Months Since First Purchase")
plt.ylabel("Cohort Month")
plt.tight_layout()

plt.show()

<Figure size 1200x700 with 2 Axes>

In [8]:
cohort_retention.to_csv("../data/cohort_retention.csv")

print("Cohort retention analysis saved successfully.")
print("File created:")
print("../data/cohort_retention.csv")

Cohort retention analysis saved successfully.
File created:
../data/cohort_retention.csv
